# 《废片之城》Q21–Q30 草图上传 GitHub

此 Notebook 只负责将聊天中已经生成的 **十张可审阅草图** 从 ZIP 上传到 `feipian/drafts/`，无需 GPU、无需下载 Qwen 模型。**注意：这些是镜头预览图，不是经 Qwen 模型生成并验收的最终分镜图。** 下载聊天提供的 `Feipian_Q21_Q30_ReviewDrafts.zip`，运行下面一格上传 ZIP 即可。GITHUB_TOKEN 仍从 Colab Secrets 获取，不能粘贴到聊天。

In [ ]:
import base64, hashlib, io, json, re, time, zipfile
from pathlib import Path
from google.colab import files as colab_files, userdata
from PIL import Image
import requests

REPO="mjw3177109/qwen2.1tu"
BRANCH="main"
token=(userdata.get("GITHUB_TOKEN") or "").strip()
if not token:
    raise RuntimeError("Colab Secrets 缺少 GITHUB_TOKEN")
headers={"Authorization":f"Bearer {token}","Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
probe=requests.get(f"https://api.github.com/repos/{REPO}",headers=headers,timeout=20)
probe.raise_for_status()
print("✅ GitHub 仓库连接正常:",probe.json()["full_name"])
print("请从本对话下载 Feipian_Q21_Q30_ReviewDrafts.zip，然后在此上传。")
upload=colab_files.upload()
if len(upload)!=1:
    raise ValueError("仅允许上传一份预览图 ZIP")
filename,body=next(iter(upload.items()))
if not filename.lower().endswith(".zip") or len(body)>120_000_000:
    raise ValueError("只接受小于120MB的ZIP")
z=zipfile.ZipFile(io.BytesIO(body))
names=z.namelist()
if any(name.startswith("/") or ".." in name.replace(chr(92),"/").split("/") for name in names):
    raise ValueError("ZIP路径不安全")
manifest=json.loads(z.read("feipian/DRAFT_MANIFEST.json"))
expected={entry["name"]:entry for entry in manifest["files"]}
if len(expected)!=10 or len(set(expected))!=10:
    raise ValueError("必须正好包含 Q21-Q30 十张独立图片")
if manifest["status"]!="LOCAL_REVIEW_DRAFTS_NOT_GITHUB_UPLOADED":
    raise ValueError("审核草图状态异常")
api=f"https://api.github.com/repos/{REPO}/contents"
for entry in manifest["files"]:
    name=entry["name"]
    if not re.fullmatch(r"Q(?:2[1-9]|30)_[A-Z0-9_]+[.]png",name):
        raise ValueError(f"不允许的文件名: {name}")
    raw=z.read("feipian/"+name)
    if hashlib.sha256(raw).hexdigest()!=entry["sha256"]:
        raise ValueError(f"图片SHA256不匹配: {name}")
    with Image.open(io.BytesIO(raw)) as image:
        image.verify()
    dest=f"feipian/drafts/{name}"
    r=requests.get(f"{api}/{dest}",headers=headers,params={"ref":BRANCH},timeout=30)
    if r.status_code==200:
        if r.json().get("sha")==hashlib.sha1(b"blob "+str(len(raw)).encode()+b"\x00"+raw).hexdigest():
            print("⏭ 已存在且一致",name)
            continue
        raise RuntimeError("已有不同的草图，拒绝覆盖："+dest)
    if r.status_code!=404:raise RuntimeError(f"读取{dest}失败 HTTP {r.status_code}")
    payload={"message":f"feipian drafts: upload {name} (not final QA)",
             "content":base64.b64encode(raw).decode("ascii"),"branch":BRANCH}
    w=requests.put(f"{api}/{dest}",headers=headers,json=payload,timeout=140)
    if w.status_code not in (200,201):raise RuntimeError(f"上传{name}失败 HTTP {w.status_code} {w.text[:180]}")
    print("✅ 上传未验收草图:",f"https://github.com/{REPO}/blob/main/{dest}")
    time.sleep(0.4)
# Create review manifest in GitHub separately.
dest="feipian/drafts/DRAFT_MANIFEST.json"
manifest["status"]="DRAFTS_UPLOADED_TO_GITHUB_PENDING_VISUAL_QA"
raw=(json.dumps(manifest,ensure_ascii=False,indent=2)+"\n").encode("utf-8")
r=requests.get(f"{api}/{dest}",headers=headers,params={"ref":BRANCH},timeout=30)
payload={"message":"feipian drafts: add QA disclaimer and source record",
         "content":base64.b64encode(raw).decode("ascii"),"branch":BRANCH}
if r.status_code==200:payload["sha"]=r.json()["sha"]
elif r.status_code!=404:raise RuntimeError(f"清单读取失败 HTTP {r.status_code}")
w=requests.put(f"{api}/{dest}",headers=headers,json=payload,timeout=50)
w.raise_for_status()
print("✅ 全部审阅素材上传完成。仅为草图，不是最终 Qwen 生成成品。")
